# Construcción del modelo base y contraste con otros modelos de regresión
Se espera construir un modelo de regresión logística para predecir la probabilidad de morosidad (`BAD = 1`) y luego contrastarlo con otros modelos basados en árboles (como `HistGradientBoosting` / LightGBM, `RandomForest` y `XGBoost`).

Para evitar fuga de información (*data leakage*), este cuaderno parte del CSV original, aplica únicamente el filtrado determinístico de filas con alta ausencia de datos documentado en `01-loadTransform.ipynb` y separa entrenamiento y prueba de forma estratificada antes de ajustar imputaciones, escalado, ingeniería de características o modelos.

## 1. Bibliotecas, rutas y configuración
Importamos las librerías estándar del proyecto (`pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy`, `scikit-learn`, `xgboost` y `cloudpickle`) y comprobamos de forma segura si `lightgbm` o `shap` están disponibles en el entorno actual.

Mantenemos exactamente la misma convención de rutas relativas como cadenas de texto desde `notebooks/` y la semilla `semilla = 865` de `01-loadTransform.ipynb`.

In [7]:
from importlib.util import find_spec
from subprocess import CREATE_NO_WINDOW, run
import warnings

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import ks_2samp
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
semilla = 865

## 2. Carga, filtro determinístico y `train_test_split` sin fuga de información
Replicamos exactamente los dos filtros determinísticos por fila definidos en `01-loadTransform.ipynb` sobre las ocho variables acordadas (`VALUE`, `CLAGE`, `NINQ`, `CLNO`, `YOJ`, `DEROG`, `DELINQ`, `DEBTINC`):
1. Se eliminan las 4 filas con las ocho variables nulas simultáneamente.
2. Se eliminan las 279 filas adicionales con 4 o más nulos entre esas ocho variables.

A continuación definimos `X` y `y = BAD`, conservando el índice original en `ID_FILA` aparte (fuera de `X`) para trazabilidad, y dividimos en **80 % entrenamiento / 20 % prueba** con `stratify=y`, `shuffle=True` y `random_state=865`.

In [9]:
datita_original = pd.read_csv('../input/hmeq.csv')

columnas_eliminacion = ['VALUE', 'CLAGE', 'NINQ', 'CLNO', 'YOJ', 'DEROG', 'DELINQ', 'DEBTINC']
filtro_1 = datita_original[columnas_eliminacion].isna().all(axis=1)
datita_intermedia = datita_original.loc[~filtro_1].copy(deep=True)
filtro_2 = datita_intermedia[columnas_eliminacion].isna().sum(axis=1).ge(4)
datita_base = datita_intermedia.loc[~filtro_2].copy(deep=True)

assert len(datita_base) == 5677

id_fila = datita_base.index.to_series(name='ID_FILA')
y = datita_base['BAD'].astype('int8')
X = datita_base.drop(columns=['BAD']).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=semilla,
    shuffle=True,
)

id_test = id_fila.loc[X_test.index]

resumen_split = pd.DataFrame({
    'filas': [len(X_train), len(X_test), len(X)],
    'porcentaje': [len(X_train) / len(X) * 100, len(X_test) / len(X) * 100, 100.0],
    'casos_bad_1': [int(y_train.sum()), int(y_test.sum()), int(y.sum())],
    'tasa_bad_pct': [y_train.mean() * 100, y_test.mean() * 100, y.mean() * 100],
}, index=['Entrenamiento (train)', 'Prueba (test)', 'Total conservado'])

display(resumen_split.round(3))

,filas,porcentaje,casos_bad_1,tasa_bad_pct
Entrenamiento (train),4541,79.989,905,19.930
Prueba (test),1136,20.011,226,19.894
Total conservado,5677,100.000,1131,19.922


### Codificación de variables categóritcas (TODO)